In [ ]:
# NOTEBOOK 34 - step 1b: is there a wind-dependent signal from the regional stations into Delhi?  (no model is trained)
# Question: when a regional station reads HIGHER than Delhi right now, does Delhi tend to rise over the next few hours -
# and is that more true when the wind is blowing from that station toward Delhi than when it is not?
#   E       = log PM2.5 at the regional station minus log PM2.5 of the Delhi city mean, now ("regional excess")
#   target  = change of the Delhi city mean (log PM2.5) over the next 3 / 6 / 12 hours
#   strata  = wind at the regional station along the line to Delhi:  aligned (>= +1 m/s toward Delhi), neutral, opposed (<= -1 m/s)
#   partial correlation of E with the target, after removing Delhi's own level and its own recent change (so plain persistence and
#   mean reversion are not mistaken for transport). Uncertainty by resampling whole days (the hours of a day are not independent).
#   Also a lag-aware version: the regional PM2.5 at the time the air left the station (travel time = distance / speed, capped at 12 h) minus Delhi NOW.
# Winter months only (Oct-Mar). Needs v2_clean/regional_cache/regional_dataset.npz from notebook 33. Reads only; writes nothing.
# HOW TO CHECK: this cell only imports and loads the data.
import os, numpy as np, pandas as pd
V2_DIR = r'D:\FYP\Eco-Twin\v2_clean'
z = np.load(os.path.join(V2_DIR, 'regional_cache', 'regional_dataset.npz'))
PM, UV, LAT, LON, NAMES, DIST = z['pm'], z['era5'], z['lat'], z['lon'], z['names'], z['dist_km']
GRID = pd.DatetimeIndex(z['grid']); T, R = PM.shape
ds = pd.read_csv(os.path.join(V2_DIR, 'EcoTwin_Final_Dataset.csv'), parse_dates=['Timestamp'])
DEL = np.log1p(ds.pivot_table(index='Timestamp', columns='Station', values='pm2.5').reindex(GRID).mean(axis=1).values)
DELHI = (28.64, 77.17)
# unit vector from each regional station toward Delhi (east, north)
de = (DELHI[1] - LON) * np.cos(np.radians(LAT)); dn = DELHI[0] - LAT; nr = np.hypot(de, dn); VEC = np.stack([de / nr, dn / nr], axis=1)
ALIGN = UV[:, :, 0] * VEC[None, :, 0] + UV[:, :, 1] * VEC[None, :, 1]          # m/s of wind blowing from the station toward Delhi (negative = away)
winter = np.isin(GRID.month, [10, 11, 12, 1, 2, 3]); day = np.array((GRID - GRID[0]).days)
print(R, 'regional stations |', T, 'hours | winter hours', int(winter.sum()))

In [ ]:
# build the observations: one row per (hour, station)
def build(h, lagged=False):
    rows = []
    for i in range(R):
        lr = np.log1p(PM[:, i])
        if lagged:
            lag = np.clip(np.rint(DIST[i] / (np.maximum(ALIGN[:, i], 0.5) * 3.6)), 0, 12).astype(int)
            idx = np.clip(np.arange(T) - lag, 0, T - 1); excess = lr[idx] - DEL                # regional PM2.5 when that air left the station, compared with Delhi NOW
        else:
            excess = lr - DEL
        t0 = np.arange(24, T - h)
        tgt = DEL[t0 + h] - DEL[t0]; lvl = DEL[t0]; past = DEL[t0] - DEL[t0 - 6]
        al = ALIGN[t0, i]; ok = np.isfinite(excess[t0]) & np.isfinite(tgt) & np.isfinite(lvl) & np.isfinite(past) & winter[t0]
        rows.append(pd.DataFrame({'st': i, 'excess': excess[t0][ok], 'tgt': tgt[ok], 'lvl': lvl[ok], 'past': past[ok], 'align': al[ok], 'day': day[t0][ok]}))
    return pd.concat(rows, ignore_index=True)

def partial_r(d):
    X = np.column_stack([np.ones(len(d)), d['lvl'].values, d['past'].values])
    res = lambda y: y - X @ np.linalg.lstsq(X, y, rcond=None)[0]
    a, b = res(d['tgt'].values), res(d['excess'].values)
    return float(np.corrcoef(a, b)[0, 1]) if len(d) > 200 else np.nan

def boot(d, n=200, seed=0):
    rng = np.random.default_rng(seed); days = np.unique(d['day']); grp = {k: g for k, g in d.groupby('day')}
    out = []
    for _ in range(n):
        s = rng.choice(days, len(days)); out.append(partial_r(pd.concat([grp[k] for k in s])))
    return np.nanpercentile(out, [2.5, 97.5]), np.array(out)

def strata(d):
    return {'aligned (>= +1 m/s toward Delhi)': d[d['align'] >= 1.0], 'neutral': d[(d['align'] > -1.0) & (d['align'] < 1.0)], 'opposed (<= -1 m/s)': d[d['align'] <= -1.0]}

summary = {}
for h in (3, 6, 12):
    d = build(h); dl = build(h, lagged=True)
    print(f'===== Delhi change over the next {h} hours =====')
    print(f"{'stratum':36}{'hours':>8}{'partial r, same-hour excess':>30}{'partial r, lag-aware excess':>30}")
    st_ = strata(d); stl = strata(dl); bs = {}
    for k in st_:
        r1 = partial_r(st_[k]); ci1, b1 = boot(st_[k]); r2 = partial_r(stl[k]); ci2, b2 = boot(stl[k]); bs[k] = (b1, b2)
        print(f'{k:36}{len(st_[k]):>8}{r1:>+14.3f}  [{ci1[0]:+.3f}, {ci1[1]:+.3f}]{r2:>+10.3f}  [{ci2[0]:+.3f}, {ci2[1]:+.3f}]')
    ka, ko = 'aligned (>= +1 m/s toward Delhi)', 'opposed (<= -1 m/s)'
    d1 = bs[ka][0] - bs[ko][0]; d2 = bs[ka][1] - bs[ko][1]
    print(f'  aligned minus opposed, same-hour excess: {np.nanmean(d1):+.3f}  95% interval [{np.nanpercentile(d1, 2.5):+.3f}, {np.nanpercentile(d1, 97.5):+.3f}]')
    print(f'  aligned minus opposed, lag-aware excess: {np.nanmean(d2):+.3f}  95% interval [{np.nanpercentile(d2, 2.5):+.3f}, {np.nanpercentile(d2, 97.5):+.3f}]')
    print(f'  lag-aware minus same-hour, aligned only: {np.nanmean(bs[ka][1] - bs[ka][0]):+.3f}')
    print()

# HOW TO READ: transport shows up as a clearly positive partial r in the ALIGNED stratum that is larger than in the OPPOSED stratum, with the
# "aligned minus opposed" interval above zero. If aligned is not above opposed (interval includes zero), the wind direction adds nothing beyond a
# plain regional average, and a wind-aware regional graph is unlikely to beat a static one. A partial r of 0.05 is small, 0.15 is meaningful.